# 📋 04 — Rekomendasi Bisnis Strategis per Cluster

**Proyek:** Segmentasi UMKM Kota Bandung menggunakan K-Means Clustering

**Tujuan Notebook Ini:**
- Membuat profil deskriptif setiap segmen cluster
- Menghasilkan rekomendasi bisnis strategis per cluster
- Menyimpan dataset akhir yang lengkap dengan segmentasi dan rekomendasi

> **Input:** `data/processed/04_Hasil_Clustering_Final.csv`
>
> **Output:** `data/processed/05_Hasil_Rekomendasi_dan_Evaluasi_LLM_V3.csv` & `data/processed/data_umkm_segmented.csv`

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

PROCESSED_PATH = os.path.join('..', 'data', 'processed')
print('Library berhasil dimuat.')

## 2. Memuat Hasil Clustering

In [ ]:
df = pd.read_csv(
    os.path.join(PROCESSED_PATH, '04_Hasil_Clustering_Final.csv'),
    sep=';', encoding='utf-8-sig'
)
print(f'Shape data: {df.shape}')
print(f'\nDistribusi Cluster:')
print(df['cluster_name'].value_counts())

## 3. Profil Deskriptif per Cluster

Melihat karakteristik rata-rata setiap segmen UMKM.

In [ ]:
profile_cols = ['totalScore', 'reviewsCount', 'sentiment_score']

for cluster_id in sorted(df['cluster'].unique()):
    cluster_data = df[df['cluster'] == cluster_id]
    name = cluster_data['cluster_name'].iloc[0]
    
    print(f'\n{"=" * 60}')
    print(f'  {name}')
    print(f'{"=" * 60}')
    print(f'  Jumlah UMKM     : {len(cluster_data):,}')
    print(f'  Rating Rerata   : {cluster_data["totalScore"].mean():.2f}')
    print(f'  Ulasan Rerata   : {cluster_data["reviewsCount"].mean():.0f}')
    print(f'  Ulasan Median   : {cluster_data["reviewsCount"].median():.0f}')
    print(f'  Sentimen Rerata : {cluster_data["sentiment_score"].mean():.4f}')
    
    print(f'\n  Top 5 Kategori:')
    if 'categoryName' in cluster_data.columns:
        top_cats = cluster_data['categoryName'].value_counts().head(5)
        for cat, cnt in top_cats.items():
            print(f'    - {cat}: {cnt}')
    
    print(f'\n  Contoh UMKM:')
    sample = cluster_data[['title', 'totalScore', 'reviewsCount']].head(3)
    for _, row in sample.iterrows():
        print(f'    - {row["title"]} (Rating: {row["totalScore"]:.1f}, Ulasan: {int(row["reviewsCount"])})')

## 4. Rekomendasi Bisnis Strategis per Cluster

Rekomendasi disesuaikan dengan hasil penelitian analisis akhir:

In [ ]:
rekomendasi_map = {
    0: 'Jaga kualitas yang telah memperoleh penilaian positif, pastikan profil akurat, dan dorong ulasan secara etis dari pelanggan asli tanpa imbalan.',
    1: 'Lakukan pemeriksaan manual tema keluhan sebelum perubahan layanan. Keputusan operasional tidak boleh hanya dari label sentimen otomatis.',
    2: 'Jaga konsistensi informasi dan pengalaman pelanggan, pantau perubahan ulasan, dan gunakan masukan sebagai bahan perbaikan berkelanjutan.'
}

df['rekomendasi_strategis'] = df['cluster'].map(rekomendasi_map)

# Tampilkan rekomendasi per cluster
print('=== REKOMENDASI STRATEGIS PER CLUSTER ===')
for cluster_id, rekomendasi in rekomendasi_map.items():
    name = df[df['cluster'] == cluster_id]['cluster_name'].iloc[0]
    print(f'\n[{name}]')
    print(f'  Rekomendasi: {rekomendasi}')

## 5. Contoh Hasil Akhir

In [ ]:
print('=== CONTOH HASIL SEGMENTASI & REKOMENDASI ===')
preview = df[['title', 'totalScore', 'reviewsCount', 'sentiment_score', 'cluster_name', 'rekomendasi_strategis']].head(5)
for _, row in preview.iterrows():
    print(f'\n  UMKM       : {row["title"]}')
    print(f'  Rating     : {row["totalScore"]:.1f}')
    print(f'  Ulasan     : {int(row["reviewsCount"])}')
    print(f'  Sentimen   : {row["sentiment_score"]:.4f}')
    print(f'  Kluster    : {row["cluster_name"]}')
    print(f'  Rekomendasi: {row["rekomendasi_strategis"]}')

## 6. Simpan Hasil Akhir

In [ ]:
# Simpan file rekomendasi
out_rekom = '05_Hasil_Rekomendasi_dan_Evaluasi_LLM_V3.csv'
df.to_csv(
    os.path.join(PROCESSED_PATH, out_rekom),
    index=False, sep=';', encoding='utf-8-sig'
)
print(f'Tersimpan: {out_rekom}')

# Simpan dataset master akhir
out_final = 'data_umkm_segmented.csv'
df.to_csv(
    os.path.join(PROCESSED_PATH, out_final),
    index=False, sep=';', encoding='utf-8-sig'
)
print(f'Tersimpan: {out_final}')

print(f'\nTotal UMKM tersegmentasi: {len(df):,}')
print(f'Jumlah kolom: {len(df.columns)}')
print(f'\n✅ Pipeline Data Mining selesai!')

## 7. Ringkasan Pipeline Lengkap

| Tahap | Notebook | Input | Output |
|-------|----------|-------|--------|
| EDA | `01_EDA_Data_Understanding.ipynb` | 19 CSV mentah | — (visualisasi saja) |
| Preprocessing | `02_Preprocessing.ipynb` | 19 CSV mentah | `03_Data_Modeling_Setelah_NLP.csv` |
| Modeling | `03_Modeling_KMeans.ipynb` | Data setelah NLP | `04_Hasil_Clustering_Final.csv` |
| Rekomendasi | `04_Rekomendasi_Bisnis.ipynb` | Hasil clustering | `data_umkm_segmented.csv` |